# Which tests should run first? Rebuild every number

This notebook rebuilds all tables and figures in the paper from the saved data: planted bugs, kill matrices, per-test runtimes, the rankings of every method, and the cached Gemma 4 answers. It runs on CPU and calls no model.

The analysis plan was frozen before any method ran: `analysis_plan.md`, SHA-256 `941004144a033e7b4edb5a011ad0ff34ab65da29c7ca4eaecf6deab4b74ca9fd`.

In [ ]:
!git clone -q https://github.com/kinnuworks/which-tests-first.git
%cd which-tests-first
!pip -q install networkx matplotlib

## Check the analysis plan hash

In [ ]:
import hashlib
for f in ('analysis_plan.md', 'analysis_plan_addendum_v2.md'):
    print(f, hashlib.sha256(open(f,'rb').read()).hexdigest())

## Tables (Recall@k with 95% bootstrap intervals, by repository, paired difference, time to first failure, static misses)

In [ ]:
!python rts/analyze.py data cachetools toolz humanize python-sortedcontainers fastapi
!python rts/analyze_real.py data/real FastAPI
!python rts/analyze_real.py data/real_rich Rich
!python rts/analyze_real.py data/real_requests requests
!python rts/analyze_real_pooled.py data
!python rts/analyze_v2_planted.py data
!python rts/analyze_encoding.py data
!python rts/analyze_memorisation.py data

## Figures

In [ ]:
!python rts/figures.py data paper/figs
!python rts/figure3.py data paper/figs
from IPython.display import Image, display
for f in ('fig1_recall_at_k', 'fig2_time_to_failure', 'fig3_planted_vs_real'):
    display(Image(f'paper/figs/{f}.png'))

## Look at one model answer

Every prompt and answer is stored in `data/rr/cache`, keyed by the SHA-256 of model name and prompt.

In [ ]:
import json, glob
f = sorted(glob.glob('data/rr/cache/*.json'))[0]
print(json.load(open(f))['content'])